<a href="https://colab.research.google.com/github/jegazhu/data-orchestrator/blob/main/Data_Orchestrator_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workbook Cell Functionality Overview

This section provides a quick guide to understanding the purpose and functionality of each major cell in this Google Colab workbook.

### Core Data Pipeline Stages:

*   **Cell `UC97s3uZjiJh` (Data Orchestrator - EVC Pipeline)**
    *   **Purpose**: This is the central hub of the data pipeline. It orchestrates the entire Extract, Validate, and Cleanse (EVC) process.
    *   **Functionality**:
        *   **Setup**: Installs necessary libraries (`kagglehub`, `pandas`, `requests`), configures robust logging (to both console and file), and creates the local directory structure for raw, transformed, validated data, and logs.
        *   **Extraction**: Defines data sources (web CSVs, Kaggle datasets) and functions (`extract_csv`, `extract_kaggle`) to pull data into pandas DataFrames.
        *   **Validation**: Implements a series of checks (`check_schema`, `check_completeness`, `check_business_rules`, etc.) to ensure data quality and integrity, generating detailed validation reports.
        *   **Cleansing**: Applies data transformation rules (`clean_drop_high_null`, `clean_fill_missing`, `clean_standardize_types`) to prepare data for loading, and generates a `README.md` summarizing changes.
        *   **Orchestration**: The `run_pipeline` function calls the extraction, validation, and cleansing stages in sequence, ensuring a controlled and logged data flow.

*   **Cell `qT-0vBQajiJk` (Log Analyzer)**
    *   **Purpose**: To analyze the logs generated by the Data Orchestrator (Cell `UC97s3uZjiJh`) and provide a prioritized summary of events.
    *   **Functionality**:
        *   **Log Parsing**: Reads the orchestrator's log file, parsing each line for timestamp, level, and message.
        *   **Priority Assignment**: Uses predefined rules (based on log level and keywords like 'error', 'warning', 'success') to assign a priority (High, Medium, Low) to each log entry.
        *   **Reporting**: Generates a summary of priorities and lists the most critical log entries with their full raw lines, saving a detailed report as a CSV file.

### SQL Server Integration Stages:

*   **Cell `O8vCCQgfjiJo` (ODBC Driver Install)**
    *   **Purpose**: To install the necessary Microsoft ODBC Driver for SQL Server in the Kaggle/Colab environment, allowing Python to connect to SQL Server databases.
    *   **Functionality**: Executes shell commands (`apt-get`, `curl`, `dpkg`) to install `unixodbc`, `msodbcsql17`, and `mssql-tools`, ensuring the driver is properly configured.

*   **Cell `vBa6kWOvjiJm` (TCP Connection Test)**
    *   **Purpose**: To quickly verify network connectivity to the SQL Server (or ngrok tunnel) endpoint before attempting a full database connection.
    *   **Functionality**: Uses Python's `socket` module to attempt a TCP connection to the specified host and port, reporting success or failure.

*   **Cell `KVnctgMCjiJn` (SQL Load - Multi-table pymssql)**
    *   **Purpose**: To load multiple cleaned CSV files (from the `transformations` directory) into a SQL Server database using the `pymssql` library.
    *   **Functionality**:
        *   **Configuration**: Defines SQL Server connection details (`SQL_CONFIG`) and a list of tables to load, including their corresponding CSV files and an option to drop existing tables.
        *   **Table Management**: Infers SQL data types from DataFrame columns, generates `CREATE TABLE` SQL statements, and handles dropping/creating tables in the database.
        *   **Batch Insertion**: Inserts data from pandas DataFrames into SQL Server tables in configurable batches to improve performance.
        *   **Diagnostic**: Includes a TCP connection test and provides feedback on the loading process and final row counts.

*   **Cell `1Zq5AuALjiJp` (SQL Load - pyodbc with Diagnostics)**
    *   **Purpose**: An alternative SQL load mechanism, focused on loading a single table with enhanced diagnostics, using the `pyodbc` library.
    *   **Functionality**: Similar to the `pymssql` loader but uses `pyodbc` for database interaction. It's often used for debugging connectivity and data type issues due to its direct interaction with ODBC drivers. It allows specific control over `TABLE_NAME`, `CSV_PATH`, and whether to perform data insertion.

*   **Cell `usiFPKXGjiJq` (SQL Load - pyodbc with Row Limit)**
    *   **Purpose**: Designed for efficiently loading data into SQL Server using `pyodbc`, particularly useful for large datasets where only a subset of rows is needed for testing or development.
    *   **Functionality**:
        *   **Row Limiting**: Allows specifying a `max_rows` parameter to load only a certain number of rows from the source CSV, which is beneficial for performance and resource management during development.
        *   **Multi-table Support**: Configured to process a list of tables, similar to the `pymssql` version, but with the added `max_rows` capability for each.
        *   **Batch Insertion**: Also performs data insertion in batches, similar to other SQL load cells.

### Initial Setup & Exploration:

*   **Cell `TfV54BJnjiJc` (KaggleHub Import)**
    *   **Purpose**: Initiates the import of Kaggle datasets linked to the notebook environment.
    *   **Functionality**: Uses `kagglehub.dataset_download()` to make Kaggle datasets available in the `/kaggle/working` directory.

*   **Cell `Uiu26INSjiJd` (Kaggle Environment Setup)**
    *   **Purpose**: Standard boilerplate code often found in Kaggle notebooks for environment setup and basic data exploration.
    *   **Functionality**: Imports common libraries like `numpy` and `pandas`, and lists files in the `/kaggle/input` directory, which typically contains competition data.

*   **Cell `GZnoU-7YjiJf` (Directory Structure Check)**
    *   **Purpose**: Verifies that the expected directory structure (`raw`, `transformations`, `validation`, `logs`) exists within the working directory (`/kaggle/working/data`).
    *   **Functionality**: Prints the status (exists or missing) for each required subdirectory, providing an early indication if the environment is set up correctly for the pipeline.

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
moneystore_agencyperformance_path = kagglehub.dataset_download('moneystore/agencyperformance')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moneystore/agencyperformance/finalapi.csv


In [ ]:
#!ls -la /kaggle/working/data
import os
base = "/kaggle/working/data"   # adjust if you used a different BASE_DIR
subdirs = ["raw", "transformations", "validation", "logs"]
for sub in subdirs:
    path = os.path.join(base, sub)
    print(f"{path}: {'exists' if os.path.isdir(path) else 'missing'}")

/kaggle/working/data/raw: missing
/kaggle/working/data/transformations: missing
/kaggle/working/data/validation: missing
/kaggle/working/data/logs: missing


In [ ]:
# =============================================================================
# DATA ORCHESTRATOR – All Stages (with robust logging)
# For Google Colab
# =============================================================================

# -----------------------------
# 1. Setup & Dependencies
# -----------------------------
!pip install -q kagglehub pandas requests

import os
import sys
import logging
import json
import pandas as pd
import requests
import kagglehub
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
import numpy as np

# Create directory structure
BASE_DIR = "/kaggle/working/data"
os.makedirs(f"{BASE_DIR}/raw", exist_ok=True)
os.makedirs(f"{BASE_DIR}/transformations", exist_ok=True)
os.makedirs(f"{BASE_DIR}/validation", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

# -------- Custom log file name --------
now = datetime.now()
log_filename = f"orchestrator{now.strftime('%Y%m%d')}:{now.strftime('%H:%M:%S')}.log"
log_path = os.path.join(BASE_DIR, "logs", log_filename)

# -------- Robust logging setup --------
logger = logging.getLogger('orchestrator')
logger.setLevel(logging.DEBUG)  # capture everything

# Remove any existing handlers (to avoid duplication)
logger.handlers.clear()

# File handler – writes to our log file
fh = logging.FileHandler(log_path)
fh.setLevel(logging.DEBUG)

# Console handler – prints to Colab output
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.DEBUG)

# Formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)

logger.addHandler(fh)
logger.addHandler(ch)

# Test entry – this ensures the file is written immediately
logger.info(f"Logging initialized. Log file: {log_path}")
# Force flush to disk (optional but safe)
fh.flush()

# =============================================================================
# STAGE 1: DATA EXTRACTION
# =============================================================================

# -------- Source configuration --------
SOURCES = [
    {
        "id": "institutions",
        "type": "csv",
        "url": "https://s3-us-gov-west-1.amazonaws.com/cg-2e5c99a6-e282-42bf-9844-35f5430338a5/downloads/institutions.csv",
        "filename": "institutions.csv",
        "options": {}
    },
    {
        "id": "household_income",
        "type": "csv",
        "url": "https://datahub.io/core/household-income-us-historical/_r/-/data/household-income-us-historical.csv",
        "filename": "household_income_us_historical.csv",
        "options": {}
    },
    {
        "id": "agency_performance",
        "type": "kaggle",
        "url": "moneystore/agencyperformance",
        "filename": "agency_performance.csv",
        "options": {}
    }
]

# -------- Extractor functions --------
def extract_csv(source: Dict) -> pd.DataFrame:
    url = source["url"]
    filename = source.get("filename", url.split("/")[-1])
    raw_path = f"{BASE_DIR}/raw/{filename}"

    logger.info(f"Downloading CSV from {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(raw_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    logger.info(f"Saved to {raw_path}")

    df = pd.read_csv(raw_path, **source.get("options", {}))
    return df

def extract_kaggle(source: Dict) -> pd.DataFrame:
    dataset_path = source["url"]
    logger.info(f"Downloading Kaggle dataset: {dataset_path}")
    download_path = kagglehub.dataset_download(dataset_path)
    logger.info(f"Downloaded to {download_path}")

    csv_files = [f for f in os.listdir(download_path) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {download_path}")
    csv_file = csv_files[0]
    file_path = os.path.join(download_path, csv_file)

    dest_filename = source.get("filename", csv_file)
    dest_path = f"{BASE_DIR}/raw/{dest_filename}"
    df = pd.read_csv(file_path, **source.get("options", {}))
    df.to_csv(dest_path, index=False)
    logger.info(f"Copied to {dest_path}")
    return df

# -------- Extraction orchestrator --------
def run_extraction(sources: List[Dict]) -> Dict[str, Dict]:
    registry = {}
    for src in sources:
        src_id = src["id"]
        logger.info(f"Extracting source: {src_id} (type={src['type']})")
        try:
            if src["type"] == "csv":
                df = extract_csv(src)
            elif src["type"] == "kaggle":
                df = extract_kaggle(src)
            else:
                raise ValueError(f"Unsupported source type: {src['type']}")

            registry[src_id] = {
                "df": df,
                "metadata": {
                    "source_id": src_id,
                    "type": src["type"],
                    "url": src.get("url"),
                    "filename": src.get("filename", ""),
                    "shape": df.shape,
                    "columns": list(df.columns),
                    "dtypes": df.dtypes.to_dict(),
                    "extraction_time": datetime.now().isoformat()
                }
            }
            logger.info(f"Source {src_id} extracted: {df.shape[0]} rows, {df.shape[1]} cols")
        except Exception as e:
            logger.error(f"Failed to extract {src_id}: {e}")
            raise RuntimeError(f"Extraction failed for {src_id}") from e
    return registry


# =============================================================================
# STAGE 2: DATA VALIDATION
# =============================================================================

# -------- Check definitions --------
def check_schema(df: pd.DataFrame, metadata: Dict) -> Dict:
    expected_cols = metadata.get("expected_columns", {})
    if not expected_cols:
        return {
            "check_name": "schema",
            "source_id": metadata["source_id"],
            "passed": True,
            "details": "No schema constraints defined",
            "severity": "info"
        }
    missing = [c for c in expected_cols if c not in df.columns]
    wrong_type = []
    for col, dtype in expected_cols.items():
        if col in df.columns and str(df[col].dtype) != dtype:
            wrong_type.append(f"{col}: expected {dtype}, got {df[col].dtype}")
    passed = (not missing) and (not wrong_type)
    details = {}
    if missing:
        details["missing_columns"] = missing
    if wrong_type:
        details["wrong_type"] = wrong_type
    return {
        "check_name": "schema",
        "source_id": metadata["source_id"],
        "passed": passed,
        "details": details if details else "All columns and types match",
        "severity": "error" if not passed else "info"
    }

def check_completeness(df: pd.DataFrame, metadata: Dict) -> Dict:
    null_rates = df.isnull().mean().to_dict()
    high_null = [col for col, rate in null_rates.items() if rate > 0.5]
    passed = len(high_null) == 0
    details = {
        "total_rows": len(df),
        "null_rates": null_rates,
        "high_null_columns": high_null
    }
    return {
        "check_name": "completeness",
        "source_id": metadata["source_id"],
        "passed": passed,
        "details": details,
        "severity": "warning" if high_null else "info"
    }

def check_logical_referential(df: pd.DataFrame, metadata: Dict) -> Dict:
    return {
        "check_name": "logical_referential",
        "source_id": metadata["source_id"],
        "passed": True,
        "details": "No referential constraints defined",
        "severity": "info"
    }

def check_business_rules(df: pd.DataFrame, metadata: Dict) -> Dict:
    issues = []
    if 'state' in df.columns:
        valid_states = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY']
        invalid = df[~df['state'].isin(valid_states)]['state'].unique()
        if len(invalid) > 0:
            issues.append(f"Invalid state codes: {list(invalid)}")
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'year' in c.lower()]
    for col in date_cols:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            min_date = df[col].min()
            if min_date.year < 1900:
                issues.append(f"{col} has date before 1900: {min_date}")
    passed = len(issues) == 0
    return {
        "check_name": "business_rules",
        "source_id": metadata["source_id"],
        "passed": passed,
        "details": issues if issues else "All business rules passed",
        "severity": "error" if not passed else "info"
    }

VALIDATION_CHECKS = [
    check_schema,
    check_completeness,
    check_logical_referential,
    check_business_rules
]

def run_validation(registry: Dict[str, Dict]) -> pd.DataFrame:
    all_results = []
    for source_id, entry in registry.items():
        df = entry["df"]
        metadata = entry["metadata"]
        for check_func in VALIDATION_CHECKS:
            result = check_func(df, metadata)
            all_results.append(result)

    report_df = pd.DataFrame(all_results)
    report_path = f"{BASE_DIR}/validation/validation_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    report_df.to_csv(report_path, index=False)
    logger.info(f"Validation report saved to {report_path}")

    summary = report_df.groupby(['source_id', 'severity']).size().unstack(fill_value=0)
    summary_path = f"{BASE_DIR}/validation/validation_summary.csv"
    summary.to_csv(summary_path)
    logger.info(f"Validation summary saved to {summary_path}")

    return report_df


# =============================================================================
# STAGE 3: DATA CLEANSING
# =============================================================================

def clean_drop_high_null(df: pd.DataFrame, metadata: Dict, threshold: float = 0.7) -> Tuple[pd.DataFrame, List[Dict]]:
    changes = []
    null_rates = df.isnull().mean()
    cols_to_drop = [col for col, rate in null_rates.items() if rate > threshold]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        for col in cols_to_drop:
            changes.append({
                "action": "drop_column",
                "column": col,
                "description": f"Dropped column due to {null_rates[col]:.1%} nulls (threshold {threshold:.0%})",
                "old_value": None,
                "new_value": None
            })
    return df, changes

def clean_fill_missing(df: pd.DataFrame, metadata: Dict) -> Tuple[pd.DataFrame, List[Dict]]:
    changes = []
    for col in df.columns:
        if df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                fill_val = df[col].median()
                df[col] = df[col].fillna(fill_val)
                changes.append({
                    "action": "fill_missing",
                    "column": col,
                    "description": f"Filled missing values with median ({fill_val})",
                    "old_value": "NaN",
                    "new_value": fill_val
                })
            elif pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].fillna(method='ffill')
                changes.append({
                    "action": "fill_missing",
                    "column": col,
                    "description": "Forward filled missing dates",
                    "old_value": "NaN",
                    "new_value": "ffill"
                })
            else:
                mode_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
                df[col] = df[col].fillna(mode_val)
                changes.append({
                    "action": "fill_missing",
                    "column": col,
                    "description": f"Filled missing with mode ('{mode_val}')",
                    "old_value": "NaN",
                    "new_value": mode_val
                })
    return df, changes

def clean_standardize_types(df: pd.DataFrame, metadata: Dict) -> Tuple[pd.DataFrame, List[Dict]]:
    changes = []
    for col in df.columns:
        if 'date' in col.lower() or 'time' in col.lower():
            try:
                df[col] = pd.to_datetime(df[col], errors='ignore')
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    changes.append({
                        "action": "convert_type",
                        "column": col,
                        "description": "Converted to datetime",
                        "old_value": "object/string",
                        "new_value": "datetime64"
                    })
            except:
                pass
        if df[col].dtype == 'object':
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
                if pd.api.types.is_numeric_dtype(df[col]):
                    changes.append({
                        "action": "convert_type",
                        "column": col,
                        "description": "Converted to numeric",
                        "old_value": "object",
                        "new_value": "numeric"
                    })
            except:
                pass
    return df, changes

CLEANING_STEPS = [
    clean_drop_high_null,
    clean_fill_missing,
    clean_standardize_types,
]

def run_cleansing(registry: Dict[str, Dict]) -> Dict[str, str]:
    output_files = {}
    all_readme_sections = []

    for source_id, entry in registry.items():
        df = entry["df"]
        metadata = entry["metadata"]
        logger.info(f"Cleansing source: {source_id}")

        change_log = []
        for step_func in CLEANING_STEPS:
            df, changes = step_func(df, metadata)
            change_log.extend(changes)
            logger.info(f"Applied {step_func.__name__} to {source_id}: {len(changes)} changes")

        out_filename = f"{source_id}_cleaned.csv"
        out_path = f"{BASE_DIR}/transformations/{out_filename}"
        df.to_csv(out_path, index=False)
        output_files[source_id] = out_path
        logger.info(f"Saved cleaned data to {out_path}")

        section = f"## Source: {source_id}\n"
        section += f"- Original shape: {metadata['shape']}\n"
        section += f"- Final shape: {df.shape}\n"
        section += f"- Number of changes: {len(change_log)}\n"
        if change_log:
            section += "### Changes applied:\n"
            for change in change_log:
                section += (f"  - {change['action']} on column '{change['column']}': "
                            f"{change['description']}\n")
        else:
            section += "  - No changes applied.\n"
        all_readme_sections.append(section)

    readme_path = f"{BASE_DIR}/transformations/README.md"
    with open(readme_path, "w") as f:
        f.write("# Data Cleansing Report\n\n")
        f.write(f"Generated on: {datetime.now().isoformat()}\n\n")
        f.write("\n".join(all_readme_sections))
    logger.info(f"README written to {readme_path}")

    return output_files


# =============================================================================
# MAIN ORCHESTRATOR
# =============================================================================

def run_pipeline(sources: List[Dict]) -> None:
    try:
        logger.info("=" * 60)
        logger.info("STAGE 1: EXTRACTION STARTED")
        registry = run_extraction(sources)
        logger.info("STAGE 1: EXTRACTION COMPLETED SUCCESSFULLY")

        logger.info("=" * 60)
        logger.info("STAGE 2: VALIDATION STARTED")
        validation_report = run_validation(registry)
        logger.info("STAGE 2: VALIDATION COMPLETED")

        logger.info("=" * 60)
        logger.info("STAGE 3: CLEANSING STARTED")
        output_files = run_cleansing(registry)
        logger.info("STAGE 3: CLEANSING COMPLETED")

        logger.info("=" * 60)
        logger.info("PIPELINE FINISHED SUCCESSFULLY")
        logger.info(f"Data saved under {BASE_DIR}")
        logger.info(f"Full log available at: {log_path}")

    except Exception as e:
        logger.error(f"PIPELINE FAILED: {e}")
        raise

# =============================================================================
# EXECUTE
# =============================================================================
if __name__ == "__main__":
    run_pipeline(SOURCES)

2026-07-20 13:14:17,569 - orchestrator - INFO - Logging initialized. Log file: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log


2026-07-20 13:14:17,569 - INFO - Logging initialized. Log file: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log


2026-07-20 13:14:17,577 - orchestrator - INFO - ============================================================


2026-07-20 13:14:17,577 - INFO - ============================================================


2026-07-20 13:14:17,578 - orchestrator - INFO - STAGE 1: EXTRACTION STARTED


...

2026-07-20 13:14:34,514 - INFO - PIPELINE FINISHED SUCCESSFULLY


2026-07-20 13:14:34,516 - orchestrator - INFO - Data saved under /kaggle/working/data


2026-07-20 13:14:34,516 - INFO - Data saved under /kaggle/working/data


2026-07-20 13:14:34,517 - orchestrator - INFO - Full log available at: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log


2026-07-20 13:14:34,517 - INFO - Full log available at: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log


In [ ]:
# =============================================================================
# LOG ANALYZER – Post‑orchestration Log Analysis (with full raw lines)
# For Google Colab
# =============================================================================

import re
import pandas as pd
from datetime import datetime
from typing import List, Dict, Optional
import os
import glob

# -------- Configuration --------
LOG_DIR = "/kaggle/working/data/logs"
ANALYSIS_OUTPUT_DIR = "/kaggle/working/data/logs"  # same directory

# -------- Priority assignment rules --------
LEVEL_PRIORITY = {
    'CRITICAL': 'High',
    'ERROR': 'High',
    'WARNING': 'Medium',
    'INFO': 'Low',
    'DEBUG': 'Low'
}

# Keyword rules (override level) – more comprehensive
KEYWORD_RULES = [
    (r'failed|error|exception|traceback|corrupt|missing|unexpected|terminated|abruptly', 'High'),
    (r'warning|invalid|null rate|high_null|dropped|schema mismatch|type mismatch', 'Medium'),
    (r'success|completed|saved|extracted|downloaded|finished', 'Low')
]

# -------- Helper functions --------
def parse_log_line(line: str) -> Optional[Dict]:
    """
    Parse a log line, extract timestamp, level, and message.
    Also stores the raw line for full context.
    """
    line_stripped = line.strip()
    pattern = r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) - \w+ - (\w+) - (.*)$'
    match = re.match(pattern, line_stripped)
    if match:
        timestamp_str, level, message = match.groups()
        timestamp = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S,%f')
        return {
            'timestamp': timestamp,
            'level': level,
            'message': message,
            'raw_line': line_stripped  # keep the full original line
        }
    return None

def assign_priority(level: str, message: str) -> str:
    """Assign priority based on level and keyword rules."""
    # First, check keyword rules (case-insensitive)
    for pattern, priority in KEYWORD_RULES:
        if re.search(pattern, message, re.IGNORECASE):
            return priority
    # Fallback to level-based
    return LEVEL_PRIORITY.get(level, 'Low')

def analyze_log_file(log_path: str) -> pd.DataFrame:
    """
    Read the log file, parse entries, assign priorities, return a DataFrame
    with columns: timestamp, level, message, priority, raw_line.
    """
    entries = []
    with open(log_path, 'r', encoding='utf-8') as f:
        for line in f:
            parsed = parse_log_line(line)
            if parsed:
                parsed['priority'] = assign_priority(parsed['level'], parsed['message'])
                entries.append(parsed)

    if not entries:
        raise ValueError(f"No valid log entries found in {log_path}")

    df = pd.DataFrame(entries)
    # Reorder columns for readability
    df = df[['timestamp', 'level', 'message', 'priority', 'raw_line']]
    return df

def find_latest_log_file(directory: str) -> Optional[str]:
    """Find the most recent orchestrator log file."""
    pattern = os.path.join(directory, "orchestrator*.log")
    files = glob.glob(pattern)
    if not files:
        return None
    latest = max(files, key=os.path.getmtime)
    return latest

def save_priority_report(df: pd.DataFrame, output_dir: str) -> str:
    """Save the analyzed log to a CSV file with priority breakdown."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = os.path.join(output_dir, f"log_priority_analysis_{timestamp}.csv")
    df.to_csv(output_path, index=False)
    return output_path

def print_summary(df: pd.DataFrame, num_entries: int = 10):
    """Print priority counts and show the most urgent entries with full lines."""
    summary = df['priority'].value_counts().reindex(['High', 'Medium', 'Low'], fill_value=0)
    print("\n===== LOG PRIORITY SUMMARY =====")
    for priority, count in summary.items():
        print(f"{priority:6s}: {count} entries")
    print("================================\n")

    # Show High and Medium entries with full raw lines
    high_medium = df[df['priority'].isin(['High', 'Medium'])]
    if not high_medium.empty:
        print(f"Top {num_entries} High/Medium priority entries (full log lines):")
        print("-" * 80)
        for idx, row in high_medium.head(num_entries).iterrows():
            print(f"[{row['priority']}] {row['raw_line']}")
        if len(high_medium) > num_entries:
            print(f"... and {len(high_medium) - num_entries} more High/Medium entries.")
        print("-" * 80)
    else:
        print("No High or Medium priority entries found.")

# -------- Main analysis function --------
def run_log_analysis(log_path: Optional[str] = None, output_dir: str = ANALYSIS_OUTPUT_DIR):
    """
    Analyze a orchestrator log file and produce a CSV with priority assignments.
    If log_path is None, automatically find the latest log file.
    """
    if log_path is None:
        log_path = find_latest_log_file(LOG_DIR)
        if log_path is None:
            raise FileNotFoundError(f"No orchestrator log files found in {LOG_DIR}")
        print(f"Using latest log file: {log_path}")
    else:
        if not os.path.exists(log_path):
            raise FileNotFoundError(f"Log file not found: {log_path}")

    print(f"Analyzing log file: {log_path}")
    df = analyze_log_file(log_path)
    print(f"Parsed {len(df)} log entries.")

    # Print summary and high/medium entries
    print_summary(df)

    # Save CSV report
    output_path = save_priority_report(df, output_dir)
    print(f"\nFull priority analysis saved to: {output_path}")

    return df, output_path

# =============================================================================
# EXECUTE (after orchestrator finishes)
# =============================================================================
if __name__ == "__main__":
    df, report_path = run_log_analysis()

Using latest log file: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log
Analyzing log file: /kaggle/working/data/logs/orchestrator20260720:13:14:17.log
Parsed 45 log entries.

===== LOG PRIORITY SUMMARY =====
High  : 3 entries
Medium: 3 entries
Low   : 39 entries

Top 10 High/Medium priority entries (full log lines):
--------------------------------------------------------------------------------
[Medium] 2026-07-20 13:14:27,373 - orchestrator - INFO - Applied clean_drop_high_null to institutions: 43 changes
[High] 2026-07-20 13:14:27,594 - orchestrator - INFO - Applied clean_fill_missing to institutions: 49 changes
[Medium] 2026-07-20 13:14:29,440 - orchestrator - INFO - Applied clean_drop_high_null to household_income: 0 changes
[High] 2026-07-20 13:14:29,444 - orchestrator - INFO - Applied clean_fill_missing to household_income: 0 changes
[Medium] 2026-07-20 13:14:29,520 - orchestrator - INFO - Applied clean_drop_high_null to agency_performance: 0 changes
[High] 2026-07-2

In [ ]:
import socket
# Replace with your ngrok host and port
host = '0.tcp.sa.ngrok.io'
port = 16535
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect((host, port))
    print(f"Connected to {host}:{port}")
    s.close()
except Exception as e:
    print(f"Connection failed: {e}")

Connected to 0.tcp.sa.ngrok.io:16535


In [ ]:
# =============================================================================
# STAGE 4: SQL LOAD – Multi‑table (pymssql) for Kaggle
# =============================================================================

!pip install -q pymssql

import pymssql
import pandas as pd
import os
import socket

# -------- Configuration --------
# Update these with your actual SQL Server (or ngrok) details
SQL_CONFIG = {
    'server': '0.tcp.sa.ngrok.io',   # ngrok host (no port)
    'port': 12345,                   # your current ngrok port
    'database': 'YOUR_DATABASE',
    'username': 'DBA_USER',
    'password': 'PASS_USER'
}

# -------- Table definitions --------
# Each entry: (table_name, csv_filename_in_transformations, drop_existing)
TABLES_TO_LOAD = [
    ('household_income_cleaned', 'household_income_cleaned.csv', False),
    # ('agency_performance_cleaned', 'agency_performance_cleaned.csv', False),
]

BASE_PATH = '/kaggle/working/data/transformations/'
BATCH_SIZE = 10000   # larger batch for speed

# -------- Helper functions (unchanged from earlier) --------
def test_tcp_connection(host, port, timeout=5):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        s.connect((host, port))
        s.close()
        return True
    except Exception as e:
        print(f"TCP test failed: {e}")
        return False

def infer_sql_type_from_series(series):
    if pd.api.types.is_integer_dtype(series):
        return 'INT'
    elif pd.api.types.is_float_dtype(series):
        return 'FLOAT'
    elif pd.api.types.is_datetime64_any_dtype(series):
        return 'DATETIME'
    elif pd.api.types.is_bool_dtype(series):
        return 'BIT'
    else:
        max_len = series.astype(str).str.len().max()
        if pd.isna(max_len):
            max_len = 0
        max_len = int(max_len)
        if max_len <= 4000:
            return f'NVARCHAR({max(1, max_len)})'
        else:
            return 'NVARCHAR(MAX)'

def generate_create_table_sql(table_name, df):
    columns = []
    for col in df.columns:
        sql_type = infer_sql_type_from_series(df[col])
        columns.append(f"[{col}] {sql_type}")
    columns_sql = ",\n    ".join(columns)
    return f"CREATE TABLE {table_name} (\n    {columns_sql}\n);"

def drop_table_if_exists(conn, table_name):
    cursor = conn.cursor()
    check_sql = f"""
    SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = 'dbo'
    """
    cursor.execute(check_sql)
    if cursor.fetchone()[0] > 0:
        print(f"Dropping existing table '{table_name}'...")
        cursor.execute(f"DROP TABLE {table_name}")
        conn.commit()
        print(f"Table '{table_name}' dropped.")

def table_exists(conn, table_name):
    cursor = conn.cursor()
    check_sql = f"""
    SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = 'dbo'
    """
    cursor.execute(check_sql)
    return cursor.fetchone()[0] > 0

def create_table(conn, table_name, create_sql):
    cursor = conn.cursor()
    if table_exists(conn, table_name):
        print(f"Table '{table_name}' already exists. Skipping creation.")
        return
    print(f"Creating table '{table_name}'...")
    cursor.execute(create_sql)
    conn.commit()
    print(f"Table '{table_name}' created successfully.")

def insert_data_batch(conn, table_name, df, batch_size=500):
    cursor = conn.cursor()
    columns = [f"[{col}]" for col in df.columns]
    placeholders = ', '.join(['%s' for _ in df.columns])
    insert_sql = f"INSERT INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"

    total_rows = len(df)
    for start in range(0, total_rows, batch_size):
        batch = df.iloc[start:start+batch_size]
        rows = [tuple(None if pd.isna(v) else v for v in row) for row in batch.values]
        cursor.executemany(insert_sql, rows)
        conn.commit()
        print(f"Inserted rows {start+1} to {min(start+batch_size, total_rows)}")
    print(f"Finished inserting {total_rows} rows into '{table_name}'.")

def get_row_count(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    return cursor.fetchone()[0]

def run_sql_load_for_table(table_name, csv_filename, config, drop_existing=False, batch_size=500):
    csv_path = os.path.join(BASE_PATH, csv_filename)
    print(f"\n--- Processing table: {table_name} ---")
    print(f"Loading cleaned data from: {csv_path}")
    if not os.path.exists(csv_path):
        print(f"❌ CSV file not found: {csv_path}")
        return False

    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns.")

    # Generate CREATE TABLE SQL
    create_sql = generate_create_table_sql(table_name, df)
    print("CREATE TABLE SQL:\n", create_sql)

    # Connect
    print(f"Connecting to {config['server']}:{config['port']} ...")
    try:
        conn = pymssql.connect(
            server=config['server'],
            port=config['port'],
            user=config['username'],
            password=config['password'],
            database=config['database'],
            timeout=30
        )
        print("✅ Connected successfully.")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return False

    # Drop if requested
    if drop_existing:
        drop_table_if_exists(conn, table_name)

    # Create table (if not exists)
    try:
        create_table(conn, table_name, create_sql)
    except Exception as e:
        print(f"❌ Table creation failed: {e}")
        conn.close()
        return False

    # Insert data
    try:
        insert_data_batch(conn, table_name, df, batch_size)
    except Exception as e:
        print(f"❌ Insertion failed: {e}")
        conn.close()
        return False

    # Verify row count
    count = get_row_count(conn, table_name)
    print(f"✅ Final row count in '{table_name}': {count}")

    conn.close()
    return True

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    # ---- Step 1: TCP test ----
    host = SQL_CONFIG['server']
    port = SQL_CONFIG['port']
    print(f"Testing TCP connectivity to {host}:{port} ...")
    if test_tcp_connection(host, port):
        print("✅ TCP port is reachable.")
    else:
        print("❌ TCP port not reachable. Check ngrok/firewall.")
        raise SystemExit("Cannot proceed without network connectivity.")

    # ---- Step 2: Load each table ----
    for table_name, csv_file, drop_existing in TABLES_TO_LOAD:
        success = run_sql_load_for_table(
            table_name=table_name,
            csv_filename=csv_file,
            config=SQL_CONFIG,
            drop_existing=drop_existing,
            batch_size=BATCH_SIZE
        )
        if not success:
            print(f"❌ Failed to load {table_name}. Stopping pipeline.")
            break
        print("=" * 60)

    print("✅ All tables loaded successfully.")

Testing TCP connectivity to 0.tcp.sa.ngrok.io:16535 ...
✅ TCP port is reachable.

--- Processing table: household_income_cleaned ---
Loading cleaned data from: /kaggle/working/data/transformations/household_income_cleaned.csv
Loaded 54 rows, 7 columns.
CREATE TABLE SQL:
 CREATE TABLE household_income_cleaned (
    [Year] NVARCHAR(13),
    [Number (thousands)] FLOAT,
    [Lowest] FLOAT,
    [Second] FLOAT,
    [Third] FLOAT,
    [Fourth] FLOAT,
    [Top 5 percent] FLOAT
);
Connecting to 0.tcp.sa.ngrok.io:16535 ...
✅ Connected successfully.
Creating table 'household_income_cleaned'...
Table 'household_income_cleaned' created successfully.
Inserted rows 1 to 54
Finished inserting 54 rows into 'household_income_cleaned'.
✅ Final row count in 'household_income_cleaned': 54
✅ All tables loaded successfully.


In [ ]:
# =============================================================================
# ODBC DRIVER INSTALL — Kaggle-specific version
# Kaggle notebooks only allow writes to /kaggle/working/, which is why the
# previous attempt failed at the curl download step (no writable destination).
# =============================================================================

import subprocess
import os

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)

commands = [
    "apt-get update -qq",
    "apt-get install -y -qq curl gnupg2 apt-transport-https",
    # Explicit -o path into the one directory Kaggle lets us write to
    f"curl -sSL -o {WORKDIR}/packages-microsoft-prod.deb https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb",
    f"dpkg -i {WORKDIR}/packages-microsoft-prod.deb",
    "apt-get update -qq",
    # unixodbc (runtime, includes odbcinst) — this was missing before;
    # unixodbc-dev alone only gives headers, not the odbcinst binary
    "apt-get install -y -qq unixodbc unixodbc-dev",
    "ACCEPT_EULA=Y apt-get install -y -qq msodbcsql17",
    "ACCEPT_EULA=Y apt-get install -y -qq mssql-tools",
]

for cmd in commands:
    print(f"\n>>> {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-2000:] if result.stdout else "")
    if result.returncode != 0:
        print(f"⚠ Command exited with code {result.returncode}")
        print(result.stderr[-2000:])

print("\n=== Registered ODBC drivers ===")
result = subprocess.run("odbcinst -q -d", shell=True, capture_output=True, text=True)
print(result.stdout or "(odbcinst still not found — see note below)")
print(result.stderr)


>>> apt-get update -qq


>>> apt-get install -y -qq curl gnupg2 apt-transport-https


>>> curl -sSL -o /kaggle/working/packages-microsoft-prod.deb https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb


>>> dpkg -i /kaggle/working/packages-microsoft-prod.deb
Selecting previously unselected package packages-microsoft-prod.
(Reading database ... 121030 files and directories currently installed.)
Preparing to unpack .../packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu22.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu22.04.1) ...


>>> apt-get update -qq


>>> apt-get install -y -qq unixodbc unixodbc-dev
Selecting previously unselected package unixodbc.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 

In [ ]:
# =============================================================================
# STAGE 4: SQL LOAD – Fully executable with diagnostics
# =============================================================================

# Install required packages (if not already)
!pip install -q pyodbc

import pyodbc
import pandas as pd
import os
import time
import socket
import logging

# Reuse logger if available, else set up a basic one
logger = logging.getLogger('orchestrator')
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

# -------- YOUR CONFIGURATION --------
SQL_CONFIG = {
    'server': '0.tcp.sa.ngrok.io',   # ngrok host (no port)
    'port': 12345,                   # your current ngrok port
    'database': 'YOUR_DATABASE',
    'username': 'DBA_USER',
    'password': 'PASS_USER',
    'driver': '{ODBC Driver 17 for SQL Server}'   # <-- IMPORTANT: set this
}

TABLE_NAME = 'household_income_cleaned'
CSV_PATH = '/kaggle/working/data/transformations/household_income_cleaned.csv'  # adjust if needed
INSERT_DATA = True

# -------- Helper functions --------
def test_tcp_connection(host, port, timeout=5):
    """Test if a TCP port is open."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        s.connect((host, port))
        s.close()
        return True
    except Exception as e:
        print(f"TCP test failed: {e}")
        return False

def infer_sql_type_from_series(series):
    """Infer SQL type from pandas Series."""
    if pd.api.types.is_integer_dtype(series):
        return 'INT'
    elif pd.api.types.is_float_dtype(series):
        return 'FLOAT'
    elif pd.api.types.is_datetime64_any_dtype(series):
        return 'DATETIME'
    elif pd.api.types.is_bool_dtype(series):
        return 'BIT'
    else:
        max_len = series.astype(str).str.len().max()
        if pd.isna(max_len):
            max_len = 0
        max_len = int(max_len)
        if max_len <= 4000:
            return f'NVARCHAR({max(1, max_len)})'
        else:
            return 'NVARCHAR(MAX)'

def generate_create_table_sql(table_name, df):
    columns = []
    for col in df.columns:
        sql_type = infer_sql_type_from_series(df[col])
        columns.append(f"[{col}] {sql_type}")
    columns_sql = ",\n    ".join(columns)
    create_sql = f"CREATE TABLE {table_name} (\n    {columns_sql}\n);"
    return create_sql

def create_table(conn, table_name, create_sql):
    cursor = conn.cursor()
    check_sql = f"""
    SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = 'dbo'
    """
    cursor.execute(check_sql)
    exists = cursor.fetchone()[0] > 0
    if exists:
        print(f"Table '{table_name}' already exists. Skipping creation.")
        return
    print(f"Creating table '{table_name}'...")
    cursor.execute(create_sql)
    conn.commit()
    print(f"Table '{table_name}' created successfully.")

def insert_data_batch(conn, table_name, df, batch_size=500):
    cursor = conn.cursor()
    columns = [f"[{col}]" for col in df.columns]
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f"INSERT INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"

    total_rows = len(df)
    for start in range(0, total_rows, batch_size):
        batch = df.iloc[start:start+batch_size]
        rows = [tuple(None if pd.isna(v) else v for v in row) for row in batch.values]
        cursor.executemany(insert_sql, rows)
        conn.commit()
        print(f"Inserted rows {start+1} to {min(start+batch_size, total_rows)}")
    print(f"Finished inserting {total_rows} rows into '{table_name}'.")

def run_sql_load(table_name, csv_path, config, insert=True):
    """Main load function – reads CSV, creates table, inserts data."""
    print(f"Loading cleaned data from: {csv_path}")
    if not os.path.exists(csv_path):
        print(f"❌ CSV file not found: {csv_path}")
        return
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns.")

    # Generate CREATE TABLE
    create_sql = generate_create_table_sql(table_name, df)
    print("CREATE TABLE SQL:\n", create_sql)

    # Connect
    conn_str = (
        f"DRIVER={config['driver']};"
        f"SERVER={config['server']};"
        f"DATABASE={config['database']};"
        f"UID={config['username']};"
        f"PWD={config['password']};"
        "Connect Timeout=30;"
    )
    print(f"Connecting to {config['server']}...")
    try:
        conn = pyodbc.connect(conn_str)
        print("✅ Connected successfully.")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    # Create table
    try:
        create_table(conn, table_name, create_sql)
    except Exception as e:
        print(f"❌ Table creation failed: {e}")
        conn.close()
        return

    # Insert
    if insert:
        try:
            insert_data_batch(conn, table_name, df)
        except Exception as e:
            print(f"❌ Insertion failed: {e}")
            conn.close()
            return
    else:
        print("Skipping data insertion.")

    conn.close()
    print("✅ SQL Load completed successfully.")

# =============================================================================
# EXECUTE
# =============================================================================
if __name__ == "__main__":
    # ---- First, test TCP connection ----
    host, port = SQL_CONFIG['server'].split(',')
    port = int(port)
    print(f"Testing TCP connectivity to {host}:{port} ...")
    if test_tcp_connection(host, port):
        print("✅ TCP port is reachable.")
    else:
        print("❌ TCP port not reachable. Check ngrok/firewall.")
        # Continue anyway? Better to stop.
        raise SystemExit("Cannot proceed without network connectivity.")

    # ---- Now run SQL load ----
    run_sql_load(
        table_name=TABLE_NAME,
        csv_path=CSV_PATH,
        config=SQL_CONFIG,
        insert=INSERT_DATA
    )

Testing TCP connectivity to 0.tcp.sa.ngrok.io:16535 ...
✅ TCP port is reachable.
Loading cleaned data from: /kaggle/working/data/transformations/household_income_cleaned.csv
Loaded 54 rows, 7 columns.
CREATE TABLE SQL:
 CREATE TABLE household_income_cleaned (
    [Year] NVARCHAR(13),
    [Number (thousands)] FLOAT,
    [Lowest] FLOAT,
    [Second] FLOAT,
    [Third] FLOAT,
    [Fourth] FLOAT,
    [Top 5 percent] FLOAT
);
Connecting to 0.tcp.sa.ngrok.io,16535...
✅ Connected successfully.
Creating table 'household_income_cleaned'...
Table 'household_income_cleaned' created successfully.
Inserted rows 1 to 54
Finished inserting 54 rows into 'household_income_cleaned'.
✅ SQL Load completed successfully.


In [ ]:
# =============================================================================
# STAGE 4: SQL LOAD – Limited rows (4k) for large tables
# =============================================================================

!pip install -q pyodbc

import pyodbc
import pandas as pd
import os
import socket
import logging

logger = logging.getLogger('orchestrator')
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

# -------- CONFIGURATION --------
SQL_CONFIG = {
    'server': '0.tcp.sa.ngrok.io',   # ngrok host (no port)
    'port': 12345,                   # your current ngrok port
    'database': 'YOUR_DATABASE',
    'username': 'DBA_USER',
    'password': 'PASS_USER',
    'driver': '{ODBC Driver 17 for SQL Server}'
}

# Define tables: (table_name, csv_path, drop_existing, max_rows)
# Set max_rows to None to load all rows, or an integer to limit.
TABLES_TO_LOAD = [
    ('agency_performance_cleaned', '/kaggle/working/data/transformations/agency_performance_cleaned.csv', True, 1000),
    # Add more tables as needed
]

BATCH_SIZE = 500   # insert rows in batches
INSERT_DATA = True   # set False to only create tables without data

# -------- Helper functions (unchanged) --------
def test_tcp_connection(host, port, timeout=5):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        s.connect((host, port))
        s.close()
        return True
    except Exception as e:
        print(f"TCP test failed: {e}")
        return False

def infer_sql_type_from_series(series):
    if pd.api.types.is_integer_dtype(series):
        return 'INT'
    elif pd.api.types.is_float_dtype(series):
        return 'FLOAT'
    elif pd.api.types.is_datetime64_any_dtype(series):
        return 'DATETIME'
    elif pd.api.types.is_bool_dtype(series):
        return 'BIT'
    else:
        max_len = series.astype(str).str.len().max()
        if pd.isna(max_len):
            max_len = 0
        max_len = int(max_len)
        if max_len <= 4000:
            return f'NVARCHAR({max(1, max_len)})'
        else:
            return 'NVARCHAR(MAX)'

def generate_create_table_sql(table_name, df):
    columns = []
    for col in df.columns:
        sql_type = infer_sql_type_from_series(df[col])
        columns.append(f"[{col}] {sql_type}")
    columns_sql = ",\n    ".join(columns)
    return f"CREATE TABLE {table_name} (\n    {columns_sql}\n);"

def table_exists(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"""
        SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = 'dbo'
    """)
    return cursor.fetchone()[0] > 0

def drop_table(conn, table_name):
    cursor = conn.cursor()
    if table_exists(conn, table_name):
        print(f"Dropping existing table '{table_name}'...")
        cursor.execute(f"DROP TABLE {table_name}")
        conn.commit()
        print(f"Table '{table_name}' dropped.")
    else:
        print(f"Table '{table_name}' does not exist; skipping drop.")

def create_table(conn, table_name, create_sql):
    cursor = conn.cursor()
    if table_exists(conn, table_name):
        print(f"Table '{table_name}' already exists. Skipping creation.")
        return
    print(f"Creating table '{table_name}'...")
    cursor.execute(create_sql)
    conn.commit()
    print(f"Table '{table_name}' created successfully.")

def insert_data_batch(conn, table_name, df, batch_size=500):
    cursor = conn.cursor()
    columns = [f"[{col}]" for col in df.columns]
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f"INSERT INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"

    total_rows = len(df)
    print(f"Inserting {total_rows} rows into '{table_name}' in batches of {batch_size}...")
    for start in range(0, total_rows, batch_size):
        batch = df.iloc[start:start+batch_size]
        rows = [tuple(None if pd.isna(v) else v for v in row) for row in batch.values]
        cursor.executemany(insert_sql, rows)
        conn.commit()
        print(f"  Inserted rows {start+1} to {min(start+batch_size, total_rows)}")
    print(f"✅ Finished inserting {total_rows} rows into '{table_name}'.")

def get_row_count(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    return cursor.fetchone()[0]

# -------- Main load function with max_rows support --------
def load_table(table_name, csv_path, drop_existing, max_rows=None, config=None, insert=True, batch_size=500):
    if config is None:
        config = SQL_CONFIG

    print(f"\n{'='*60}")
    print(f"Processing table: {table_name}")
    print(f"CSV path: {csv_path}")

    # Check CSV exists
    if not os.path.exists(csv_path):
        print(f"❌ CSV file not found: {csv_path}")
        return False

    # Read CSV – limit rows if max_rows is provided
    if max_rows is not None and max_rows > 0:
        print(f"Reading only first {max_rows} rows (limit set).")
        df = pd.read_csv(csv_path, nrows=max_rows)
    else:
        df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns.")

    # Generate CREATE TABLE SQL
    create_sql = generate_create_table_sql(table_name, df)
    print("CREATE TABLE SQL:\n", create_sql)

    # Connect
    conn_str = (
        f"DRIVER={config['driver']};"
        f"SERVER={config['server']};"
        f"DATABASE={config['database']};"
        f"UID={config['username']};"
        f"PWD={config['password']};"
        "Connect Timeout=30;"
    )
    print(f"Connecting to {config['server']}...")
    try:
        conn = pyodbc.connect(conn_str)
        print("✅ Connected successfully.")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return False

    # Drop if requested
    if drop_existing:
        drop_table(conn, table_name)

    # Create table
    try:
        create_table(conn, table_name, create_sql)
    except Exception as e:
        print(f"❌ Table creation failed: {e}")
        conn.close()
        return False

    # Insert
    if insert:
        try:
            insert_data_batch(conn, table_name, df, batch_size)
        except Exception as e:
            print(f"❌ Insertion failed: {e}")
            conn.close()
            return False
    else:
        print("Skipping data insertion (INSERT_DATA=False).")

    # Verify row count
    count = get_row_count(conn, table_name)
    print(f"✅ Final row count in SQL Server: {count}")

    conn.close()
    return True

# =============================================================================
# EXECUTE
# =============================================================================
if __name__ == "__main__":
    # ---- TCP connectivity test ----
    host, port = SQL_CONFIG['server'].split(',')
    port = int(port)
    print(f"Testing TCP connectivity to {host}:{port} ...")
    if test_tcp_connection(host, port):
        print("✅ TCP port is reachable.")
    else:
        print("❌ TCP port not reachable. Check ngrok/firewall.")
        raise SystemExit("Cannot proceed without network connectivity.")

    # ---- Load each table (with max_rows) ----
    for table_name, csv_path, drop_existing, max_rows in TABLES_TO_LOAD:
        success = load_table(
            table_name=table_name,
            csv_path=csv_path,
            drop_existing=drop_existing,
            max_rows=max_rows,
            config=SQL_CONFIG,
            insert=INSERT_DATA,
            batch_size=BATCH_SIZE
        )
        if not success:
            print(f"❌ Failed to load {table_name}. Stopping pipeline.")
            break
        print("=" * 60)

    print("✅ All tables processed successfully.")

Testing TCP connectivity to 0.tcp.sa.ngrok.io:16535 ...
✅ TCP port is reachable.

Processing table: agency_performance_cleaned
CSV path: /kaggle/working/data/transformations/agency_performance_cleaned.csv
Reading only first 1000 rows (limit set).
Loaded 1000 rows, 49 columns.
CREATE TABLE SQL:
 CREATE TABLE agency_performance_cleaned (
    [AGENCY_ID] INT,
    [PRIMARY_AGENCY_ID] INT,
    [PROD_ABBR] NVARCHAR(10),
    [PROD_LINE] NVARCHAR(2),
    [STATE_ABBR] NVARCHAR(2),
    [STAT_PROFILE_DATE_YEAR] NVARCHAR(29),
    [RETENTION_POLY_QTY] INT,
    [POLY_INFORCE_QTY] INT,
    [PREV_POLY_INFORCE_QTY] INT,
    [NB_WRTN_PREM_AMT] FLOAT,
    [WRTN_PREM_AMT] FLOAT,
    [PREV_WRTN_PREM_AMT] FLOAT,
    [PRD_ERND_PREM_AMT] FLOAT,
    [PRD_INCRD_LOSSES_AMT] FLOAT,
    [MONTHS] INT,
    [RETENTION_RATIO] FLOAT,
    [LOSS_RATIO] FLOAT,
    [LOSS_RATIO_3YR] FLOAT,
    [GROWTH_RATE_3YR] FLOAT,
    [AGENCY_APPOINTMENT_YEAR] INT,
    [ACTIVE_PRODUCERS] INT,
    [MAX_AGE] INT,
    [MIN_AGE] INT,
    [V